# Shadow Revenue Detection System

# Bronze Layer

## Objective

The Bronze layer is the first stage of the Medallion Architecture. In this layer, raw datasets are ingested from the managed Unity Catalog Volume into Spark DataFrames without applying any transformations. The primary objective is to preserve the original source data and store it as Delta tables for downstream processing.

## Import Required Libraries

Import the required PySpark libraries that will be used for reading datasets, performing basic profiling, and storing the data as Delta tables.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Project Configuration

Define reusable configuration variables and dataset paths. Centralizing these values makes the notebook easier to maintain and avoids hardcoding paths throughout the project.

In [0]:
# Project Configuration

CATALOG = "shadow_revenue_catalog"
BRONZE_SCHEMA = "bronze"
RAW_VOLUME = "raw_data"
RAW_DATA_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{RAW_VOLUME}"
CUSTOMERS_PATH = f"{RAW_DATA_PATH}/customers.csv"
PRODUCTS_PATH = f"{RAW_DATA_PATH}/products.csv"
ORDERS_PATH = f"{RAW_DATA_PATH}/orders.csv"
PAYMENTS_PATH = f"{RAW_DATA_PATH}/payments.csv"

## Load Raw Datasets

Read all source CSV files from the managed Volume into Spark DataFrames. Schema inference is enabled to automatically determine the appropriate data types for each column.

In [0]:
# Read Customers Dataset
customers_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CUSTOMERS_PATH)
)

# Read Products Dataset
products_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(PRODUCTS_PATH)
)

# Read Orders Dataset
orders_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ORDERS_PATH)
)

# Read Payments Dataset
payments_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(PAYMENTS_PATH)
)

## Preview Loaded Data

Display sample records from each dataset to verify that the files have been loaded successfully and to gain an initial understanding of the data.

In [0]:
# Preview Customers Dataset
display(customers_df)

# Preview Products Dataset
display(products_df)

# Preview Orders Dataset
display(orders_df)

# Preview Payments Dataset
display(payments_df)

customer_id,name,city,signup_date
1,Cust_1,Pune,2023-02-10
2,Cust_2,Jaipur,2023-10-18
3,Cust_3,Pune,2023-09-09
4,Cust_4,Pune,2023-06-08
5,Cust_5,Pune,2023-06-29
6,Cust_6,Mumbai,2022-09-14
7,Cust_7,Jaipur,2023-02-06
8,Cust_8,Pune,2022-05-11
9,Cust_9,Mumbai,2022-12-06
10,Cust_10,Mumbai,2022-03-30


product_id,price,category,effective_date,end_date,is_current
1,1823.07,Home,2023-01-01,2023-12-31,0
1,1781.59,Home,2024-01-01,null,1
2,1974.03,Home,2023-01-01,2023-12-31,0
2,1837.12,Home,2024-01-01,null,1
3,1656.6,Electronics,2023-01-01,2023-12-31,0
3,1582.66,Electronics,2024-01-01,null,1
4,803.68,Home,2023-01-01,2023-12-31,0
4,702.12,Home,2024-01-01,null,1
5,297.5,Clothing,2023-01-01,2023-12-31,0
5,368.14,Clothing,2024-01-01,null,1


order_id,product_id,customer_id,quantity,price,order_date,order_status,channel,discount
1,29,1950,4,1701.72,2024-02-29,Cancelled,Web,0.05
2,98,1491,5,1643.01,2024-01-28,Completed,Store,0.16
3,2,1518,4,1948.47,2024-01-10,Completed,Store,0.18
4,33,440,3,1911.69,2024-02-01,Completed,Web,0.07
5,1,1011,4,795.88,2024-02-11,Completed,App,0.25
6,31,741,3,625.85,2024-01-31,Completed,Store,0.15
7,35,201,3,174.15,2024-01-14,Completed,Web,0.22
8,100,490,5,823.93,2024-02-29,Cancelled,Web,0.06
9,39,269,1,1880.09,2024-02-10,Cancelled,Web,0.27
10,50,1203,3,522.32,2024-01-12,Completed,App,0.19


order_id,payment_amount,payment_method,payment_date
16228,154.69,NetBanking,2024-01-10
11388,1415.76,Wallet,2024-01-18
14772,373.19,NetBanking,2024-01-09
3002,1679.11,NetBanking,2024-02-24
8702,1604.23,Wallet,2024-01-12
2713,611.89,Wallet,2024-01-04
16472,1772.76,NetBanking,2024-02-12
3604,1125.51,Wallet,2024-01-03
16367,174.84,Wallet,2024-01-19
18286,1137.77,UPI,2024-01-17


## Inspect Dataset Schema

Review the schema of each dataset to verify column names and inferred data types before proceeding with data profiling.

In [0]:
# Print Customers Schema
customers_df.printSchema()

# Print Products Schema
products_df.printSchema()

# Print Orders Schema
orders_df.printSchema()

# Print Payments Schema
payments_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: date (nullable = true)

root
 |-- product_id: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- category: string (nullable = true)
 |-- effective_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- is_current: integer (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- discount: double (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- payment_amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_date: date (nullable = true)



## Data Profiling

Perform basic profiling on the raw datasets to understand their structure and quality. This includes checking the total number of records, missing values, and duplicate records. Since the Bronze layer preserves raw data, these checks are for assessment only and no modifications are made.

In [0]:
# Record Count
datasets = {
    "Customers": customers_df,
    "Products": products_df,
    "Orders": orders_df,
    "Payments": payments_df
}

for name, df in datasets.items():
    print(f"{name}: {df.count()} records")

Customers: 2000 records
Products: 200 records
Orders: 20400 records
Payments: 18600 records


In [0]:
# Null Value Count
for name, df in datasets.items():

    print(f"\n{name} Dataset")

    df.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in df.columns
    ]).show()


Customers Dataset
+-----------+----+----+-----------+
|customer_id|name|city|signup_date|
+-----------+----+----+-----------+
|          0|   0|   0|          0|
+-----------+----+----+-----------+


Products Dataset
+----------+-----+--------+--------------+--------+----------+
|product_id|price|category|effective_date|end_date|is_current|
+----------+-----+--------+--------------+--------+----------+
|         0|    0|       0|             0|     100|         0|
+----------+-----+--------+--------------+--------+----------+


Orders Dataset
+--------+----------+-----------+--------+-----+----------+------------+-------+--------+
|order_id|product_id|customer_id|quantity|price|order_date|order_status|channel|discount|
+--------+----------+-----------+--------+-----+----------+------------+-------+--------+
|       0|         0|          0|       0|    0|         0|           0|      0|       0|
+--------+----------+-----------+--------+-----+----------+------------+-------+--------+


In [0]:
# Duplicate Record Count
for name, df in datasets.items():

    duplicate_count = df.count() - df.dropDuplicates().count()

    print(f"{name}: {duplicate_count} duplicate records")

Customers: 0 duplicate records
Products: 0 duplicate records
Orders: 400 duplicate records
Payments: 0 duplicate records


## Store Bronze Tables

Store the raw DataFrames as Delta tables in the Bronze schema. No cleaning or transformations are applied in this layer to preserve the original source data.

In [0]:
# Save Customers Table
customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("shadow_revenue_catalog.bronze.customers")

# Save Products Table
products_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("shadow_revenue_catalog.bronze.products")

# Save Orders Table
orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("shadow_revenue_catalog.bronze.orders")

# Save Payments Table
payments_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("shadow_revenue_catalog.bronze.payments")

## Validate Bronze Tables

Verify that all Bronze Delta tables have been created successfully before moving to the Silver layer.

In [0]:
%sql
SHOW TABLES IN shadow_revenue_catalog.bronze;

database,tableName,isTemporary
bronze,customers,false
bronze,orders,false
bronze,payments,false
bronze,products,false


In [0]:
%sql
SELECT COUNT(*) AS total_orders
FROM shadow_revenue_catalog.bronze.orders;

total_orders
20400


In [0]:
%sql
SELECT *
FROM shadow_revenue_catalog.bronze.orders
LIMIT 10;

order_id,product_id,customer_id,quantity,price,order_date,order_status,channel,discount
1,29,1950,4,1701.72,2024-02-29,Cancelled,Web,0.05
2,98,1491,5,1643.01,2024-01-28,Completed,Store,0.16
3,2,1518,4,1948.47,2024-01-10,Completed,Store,0.18
4,33,440,3,1911.69,2024-02-01,Completed,Web,0.07
5,1,1011,4,795.88,2024-02-11,Completed,App,0.25
6,31,741,3,625.85,2024-01-31,Completed,Store,0.15
7,35,201,3,174.15,2024-01-14,Completed,Web,0.22
8,100,490,5,823.93,2024-02-29,Cancelled,Web,0.06
9,39,269,1,1880.09,2024-02-10,Cancelled,Web,0.27
10,50,1203,3,522.32,2024-01-12,Completed,App,0.19
